In [ ]:
# Install the dependency required for the following notebook steps.
!pip install pandas

In [ ]:
# 1. Imports
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Handle expected failures with a clear, actionable message.
try:
    import pandas as pd
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from IPython.display import display
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Required packages are missing. Install undetected-chromedriver, "
        "selenium, beautifulsoup4, and pandas in this Jupyter kernel, "
        "then restart the kernel."
    ) from exc


In [ ]:
# 2. User Parameters
matchday_input = input("Enter the Bundesliga matchday number: " ).strip()

# Handle expected failures with a clear, actionable message.
try:
    matchday = int(matchday_input)
except ValueError as exc:
    raise ValueError("Matchday must be entered as a whole number.") from exc

# Validate the input before continuing with later processing.
if matchday < 1:
    raise ValueError("Matchday must be a positive integer.")

# Edit these values if your local Chrome setup or preferred limits differ.
CHROME_MAJOR_VERSION = 150
# Set workflow configuration value: HEADLESS.
HEADLESS = False
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 30
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 20
# Set workflow configuration value: MAX_PAGES.
MAX_PAGES = 10
# Set workflow configuration value: FORM_MATCH_COUNT.
FORM_MATCH_COUNT = 5
# Set workflow configuration value: BUNDESLIGA_UNIQUE_TOURNAMENT_ID.
BUNDESLIGA_UNIQUE_TOURNAMENT_ID = 35
# Set workflow configuration value: TEAM_EVENTS_URL_TEMPLATE.
TEAM_EVENTS_URL_TEMPLATE = (
    "https://www.sofascore.com/api/v1/team/{team_id}/events/last/{page}"
)

# Process each available item while preserving the current workflow state.
for setting_name, setting_value in {
    "CHROME_MAJOR_VERSION": CHROME_MAJOR_VERSION,
    "PAGE_LOAD_TIMEOUT_SECONDS": PAGE_LOAD_TIMEOUT_SECONDS,
    "WAIT_TIMEOUT_SECONDS": WAIT_TIMEOUT_SECONDS,
    "MAX_PAGES": MAX_PAGES,
    "FORM_MATCH_COUNT": FORM_MATCH_COUNT,
}.items():
    # Validate the input before continuing with later processing.
    if not isinstance(setting_value, int) or isinstance(setting_value, bool) or setting_value < 1:
        raise ValueError(f"{setting_name} must be a positive integer.")

# Validate the input before continuing with later processing.
if FORM_MATCH_COUNT != 5:
    raise ValueError("FORM_MATCH_COUNT must remain 5 for five-match form.")


In [ ]:
# 3. Load Matchday JSON
input_path = Path.cwd() / f"match_ids_{matchday}.json"

# Handle expected failures with a clear, actionable message.
try:
    raw_fixtures = json.loads(input_path.read_text(encoding="utf-8"))
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Input file not found: {input_path}. Place the notebook beside the JSON file."
    ) from exc
except UnicodeDecodeError as exc:
    raise ValueError(f"Input file is not valid UTF-8: {input_path}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Input file is not valid JSON (line {exc.lineno}, column {exc.colno}): "
        f"{input_path}"
    ) from exc
except OSError as exc:
    raise OSError(f"Could not read input file {input_path}: {exc}") from exc

# Validate the input before continuing with later processing.
if not isinstance(raw_fixtures, list):
    raise ValueError("The input JSON must contain a top-level list of fixtures.")

required_fields = {
    "home_team",
    "home_team_id",
    "away_team",
    "away_team_id",
    "match_id",
}
teams: dict[int, dict[str, Any]] = {}

# Check whether positive integer for reuse in the workflow.
def is_positive_integer(value: Any) -> bool:
    return isinstance(value, int) and not isinstance(value, bool) and value > 0

# Process each available item while preserving the current workflow state.
for fixture_number, fixture in enumerate(raw_fixtures, start=1):
    # Validate the input before continuing with later processing.
    if not isinstance(fixture, dict):
        raise ValueError(f"Fixture {fixture_number} must be a JSON object.")

    missing_fields = required_fields.difference(fixture)
    # Validate the input before continuing with later processing.
    if missing_fields:
        missing_text = ", ".join(sorted(missing_fields))
        raise ValueError(f"Fixture {fixture_number} is missing: {missing_text}.")

    match_id = fixture["match_id"]
    # Validate the input before continuing with later processing.
    if not is_positive_integer(match_id):
        raise ValueError(f"Fixture {fixture_number} has no valid positive match_id.")

    # Process each available item while preserving the current workflow state.
    for side in ("home", "away"):
        team_name = fixture[f"{side}_team"]
        team_id = fixture[f"{side}_team_id"]

        # Validate the input before continuing with later processing.
        if not isinstance(team_name, str) or not team_name.strip():
            raise ValueError(
                f"Fixture {fixture_number} has no valid {side}_team name."
            )
        # Validate the input before continuing with later processing.
        if not is_positive_integer(team_id):
            raise ValueError(
                f"Fixture {fixture_number} has no valid positive {side}_team_id."
            )

        clean_name = team_name.strip()
        existing = teams.get(team_id)
        # Validate the input before continuing with later processing.
        if existing is not None and (
            existing["team"] != clean_name or existing["match_id"] != match_id
        ):
            raise ValueError(
                f"Team ID {team_id} appears with conflicting team or fixture data."
            )

        teams[team_id] = {"team": clean_name, "match_id": match_id}

print(f"Selected matchday: {matchday}")
print(f"Input file: {input_path}")
print(f"Loaded fixtures: {len(raw_fixtures)}")
print(f"Unique teams: {len(teams)}")


In [ ]:
# 4. Start Chrome
options = uc.ChromeOptions()
options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
if HEADLESS:
    options.add_argument("--headless=new")

driver = None
# Handle expected failures with a clear, actionable message.
try:
    driver = uc.Chrome(
        options=options,
        version_main=CHROME_MAJOR_VERSION,
        use_subprocess=True,
    )
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
except Exception as exc:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
        except Exception:
            pass
        driver = None
    raise RuntimeError(
        f"Could not initialize undetected Chrome with major version "
        f"{CHROME_MAJOR_VERSION}. Adjust CHROME_MAJOR_VERSION if needed. "
        f"Original error: {exc}"
    ) from exc

print(
    f"One reusable undetected Chrome instance is ready "
    f"(major version {CHROME_MAJOR_VERSION}, headless={HEADLESS})."
)


In [ ]:
# 5. SofaScore API Helper
class SofaScorePageError(Exception):
    """Raised when Chrome cannot return a usable SofaScore event page."""


# Retrieve team events page for reuse in the workflow.
def fetch_team_events_page(
    team_id: int,
    page: int,
) -> tuple[list[dict[str, Any]], bool]:
    # Validate the input before continuing with later processing.
    if driver is None:
        raise SofaScorePageError("Chrome is not initialized.")

    url = TEAM_EVENTS_URL_TEMPLATE.format(team_id=team_id, page=page)
    print(f"    Loading page {page}: {url}")

    # Handle expected failures with a clear, actionable message.
    try:
        driver.get(url)
    except TimeoutException as exc:
        raise SofaScorePageError(
            f"Timed out after {PAGE_LOAD_TIMEOUT_SECONDS} seconds loading {url}."
        ) from exc
    except WebDriverException as exc:
        raise SofaScorePageError(f"Chrome could not load {url}: {exc}") from exc

    # Handle expected failures with a clear, actionable message.
    try:
        WebDriverWait(driver, WAIT_TIMEOUT_SECONDS).until(
            EC.presence_of_element_located((By.TAG_NAME, "pre"))
        )
    except TimeoutException as exc:
        raise SofaScorePageError(
            f"No <pre> element appeared within {WAIT_TIMEOUT_SECONDS} seconds for {url}."
        ) from exc
    except WebDriverException as exc:
        raise SofaScorePageError(
            f"Chrome could not inspect the rendered response for {url}: {exc}"
        ) from exc

    soup = BeautifulSoup(driver.page_source, "html.parser")
    pre_tag = soup.find("pre")
    # Validate the input before continuing with later processing.
    if pre_tag is None:
        raise SofaScorePageError(f"Rendered page contains no <pre> element: {url}")

    response_text = pre_tag.get_text().strip()
    # Validate the input before continuing with later processing.
    if not response_text:
        raise SofaScorePageError(f"The <pre> element is empty: {url}")

    # Handle expected failures with a clear, actionable message.
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise SofaScorePageError(
            f"Invalid JSON for {url} (line {exc.lineno}, column {exc.colno})."
        ) from exc

    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise SofaScorePageError(f"SofaScore response is not a JSON object: {url}")

    events = payload.get("events")
    # Validate the input before continuing with later processing.
    if not isinstance(events, list):
        raise SofaScorePageError(
            f"SofaScore response has no valid events list: {url}"
        )

    has_next_page = payload.get("hasNextPage")
    if not isinstance(has_next_page, bool):
        print("    Warning: hasNextPage is missing or invalid; treating it as false.")
        has_next_page = False

    return events, has_next_page


In [ ]:
# 6. Result Parser
def nested_get(mapping: Any, *keys: str) -> Any:
    current = mapping
    # Process each available item while preserving the current workflow state.
    for key in keys:
        if not isinstance(current, dict):
            return None
        current = current.get(key)
    return current


# Check whether number for reuse in the workflow.
def is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool)


# Parse and validate finished event for reuse in the workflow.
def parse_finished_event(
    event: dict[str, Any],
    team_id: int,
    current_match_id: int,
) -> dict[str, Any] | None:
    if not isinstance(event, dict):
        print("    Warning: skipped a non-object event.")
        return None

    event_id = event.get("id")
    if not isinstance(event_id, int) or isinstance(event_id, bool):
        print("    Warning: skipped an event with no valid numeric ID.")
        return None
    if event_id == current_match_id:
        return None

    if nested_get(event, "status", "type") != "finished":
        return None

    home_team_id = nested_get(event, "homeTeam", "id")
    away_team_id = nested_get(event, "awayTeam", "id")
    if (
        not isinstance(home_team_id, int)
        or isinstance(home_team_id, bool)
        or not isinstance(away_team_id, int)
        or isinstance(away_team_id, bool)
    ):
        print(f"    Warning: skipped event {event_id}; team IDs are missing or invalid.")
        return None

    # Choose the appropriate path for the current data state.
    if team_id == home_team_id:
        analysed_team_is_home = True
    # Choose the appropriate path for the current data state.
    elif team_id == away_team_id:
        analysed_team_is_home = False
    else:
        print(
            f"    Warning: skipped event {event_id}; team ID {team_id} "
            "matches neither side."
        )
        return None

    home_score = nested_get(event, "homeScore", "normaltime")
    away_score = nested_get(event, "awayScore", "normaltime")
    if not is_number(home_score) or not is_number(away_score):
        print(
            f"    Warning: skipped event {event_id}; normal-time scores are missing."
        )
        return None

    timestamp = event.get("startTimestamp")
    if not is_number(timestamp):
        print(f"    Warning: skipped event {event_id}; startTimestamp is missing.")
        return None

    # Handle expected failures with a clear, actionable message.
    try:
        readable_date = datetime.fromtimestamp(timestamp, tz=timezone.utc).isoformat()
    except (OverflowError, OSError, ValueError) as exc:
        print(f"    Warning: skipped event {event_id}; invalid timestamp ({exc}).")
        return None

    analysed_score = home_score if analysed_team_is_home else away_score
    opponent_score = away_score if analysed_team_is_home else home_score
    # Choose the appropriate path for the current data state.
    if analysed_score > opponent_score:
        result = "W"
    # Choose the appropriate path for the current data state.
    elif analysed_score < opponent_score:
        result = "L"
    else:
        result = "D"

    return {
        "match_id": event_id,
        "date": readable_date,
        "timestamp": timestamp,
        "competition": nested_get(event, "tournament", "name"),
        "unique_tournament_id": nested_get(
            event, "tournament", "uniqueTournament", "id"
        ),
        "home_team": nested_get(event, "homeTeam", "name"),
        "home_team_id": home_team_id,
        "away_team": nested_get(event, "awayTeam", "name"),
        "away_team_id": away_team_id,
        "home_score": home_score,
        "away_score": away_score,
        "result": result,
    }


In [ ]:
# 7. Team Form Function
def select_recent_matches(
    candidates: list[dict[str, Any]],
) -> tuple[list[dict[str, Any]], str]:
    latest_matches = sorted(
        candidates,
        key=lambda match: match["timestamp"],
        reverse=True,
    )[:FORM_MATCH_COUNT]
    selected_matches = sorted(
        latest_matches,
        key=lambda match: match["timestamp"],
    )
    form = "".join(match["result"] for match in selected_matches)
    return selected_matches, form


# Collect team form for reuse in the workflow.
def collect_team_form(
    team_id: int,
    team_name: str,
    current_match_id: int,
) -> dict[str, Any]:
    overall_candidates: list[dict[str, Any]] = []
    bundesliga_candidates: list[dict[str, Any]] = []
    seen_event_ids: set[int] = set()

    # Process each available item while preserving the current workflow state.
    for page in range(MAX_PAGES):
        # Handle expected failures with a clear, actionable message.
        try:
            events, has_next_page = fetch_team_events_page(team_id, page)
        except SofaScorePageError as exc:
            print(f"    Warning: stopped pagination for {team_name}: {exc}")
            break

        if not events:
            print(f"    No events returned on page {page}; history has ended.")
            break

        # Process each available item while preserving the current workflow state.
        for event in events:
            event_id = event.get("id") if isinstance(event, dict) else None
            if isinstance(event_id, int) and not isinstance(event_id, bool):
                if event_id in seen_event_ids:
                    continue
                seen_event_ids.add(event_id)

            record = parse_finished_event(event, team_id, current_match_id)
            if record is None:
                continue

            overall_candidates.append(record)

            if (
                record["unique_tournament_id"]
                == BUNDESLIGA_UNIQUE_TOURNAMENT_ID
            ):
                unique_tournament = nested_get(
                    event, "tournament", "uniqueTournament"
                )
                unique_name = (
                    unique_tournament.get("name")
                    if isinstance(unique_tournament, dict)
                    else None
                )
                unique_slug = (
                    unique_tournament.get("slug")
                    if isinstance(unique_tournament, dict)
                    else None
                )
                if unique_name != "Bundesliga" or unique_slug != "bundesliga":
                    print(
                        f"    Warning: event {record['match_id']} has Bundesliga ID "
                        f"{BUNDESLIGA_UNIQUE_TOURNAMENT_ID} but unexpected "
                        f"name/slug values: {unique_name!r}, {unique_slug!r}."
                    )
                bundesliga_candidates.append(record)

        if (
            len(overall_candidates) >= FORM_MATCH_COUNT
            and len(bundesliga_candidates) >= FORM_MATCH_COUNT
        ):
            break

        if not has_next_page:
            print(f"    SofaScore reports no page after page {page}.")
            break
    else:
        print(f"    Reached MAX_PAGES={MAX_PAGES} for {team_name}.")

    overall_matches, overall_form = select_recent_matches(overall_candidates)
    bundesliga_matches, bundesliga_form = select_recent_matches(
        bundesliga_candidates
    )

    if len(overall_matches) < FORM_MATCH_COUNT:
        print(
            f"    Warning: {team_name} has only {len(overall_matches)} valid "
            f"overall match(es); expected {FORM_MATCH_COUNT}."
        )
    if len(bundesliga_matches) < FORM_MATCH_COUNT:
        print(
            f"    Warning: {team_name} has only {len(bundesliga_matches)} valid "
            f"Bundesliga match(es); expected {FORM_MATCH_COUNT}."
        )

    return {
        "team": team_name,
        "overall_form": overall_form,
        "bundesliga_form": bundesliga_form,
        "overall_matches": overall_matches,
        "bundesliga_matches": bundesliga_matches,
    }


In [ ]:
# 8. Process All Teams
form_results: dict[int, dict[str, Any]] = {}
total_teams = len(teams)

# Process each available item while preserving the current workflow state.
for team_number, (team_id, team_info) in enumerate(teams.items(), start=1):
    team_name = team_info["team"]
    current_match_id = team_info["match_id"]
    print(
        f"[{team_number}/{total_teams}] {team_name} "
        f"(team_id={team_id}, excluded_match_id={current_match_id})"
    )

    # Handle expected failures with a clear, actionable message.
    try:
        form_results[team_id] = collect_team_form(
            team_id=team_id,
            team_name=team_name,
            current_match_id=current_match_id,
        )
    except Exception as exc:
        print(
            f"    Unexpected team error: {type(exc).__name__}: {exc}. "
            "Continuing with the remaining teams."
        )
        form_results[team_id] = {
            "team": team_name,
            "overall_form": "",
            "bundesliga_form": "",
            "overall_matches": [],
            "bundesliga_matches": [],
        }

print("Finished processing all teams.")


In [ ]:
# 9. Final Form DataFrame
summary_rows = [
    {
        "team_id": team_id,
        "team": result["team"],
        "overall_form": result["overall_form"],
        "overall_match_count": len(result["overall_matches"]),
        "bundesliga_form": result["bundesliga_form"],
        "bundesliga_match_count": len(result["bundesliga_matches"]),
    }
    for team_id, result in form_results.items()
]

form_summary_df = pd.DataFrame(summary_rows)
display(form_summary_df)


In [ ]:
# 10. Detailed Histories
history_columns = [
    "date",
    "competition",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "result",
    "match_id",
    "timestamp",
    "unique_tournament_id",
]

# Process each available item while preserving the current workflow state.
for team_id, result in form_results.items():
    print(f"\n{result['team']} (team_id={team_id})")
    print(f"Overall form: {result['overall_form'] or '(no valid matches)'}")
    # Choose the appropriate path for the current data state.
    if result["overall_matches"]:
        display(
            pd.DataFrame(result["overall_matches"]).reindex(
                columns=history_columns
            )
        )
    else:
        print("No valid overall matches were collected.")

    print(
        f"Bundesliga form: "
        f"{result['bundesliga_form'] or '(no valid matches)'}"
    )
    # Choose the appropriate path for the current data state.
    if result["bundesliga_matches"]:
        display(
            pd.DataFrame(result["bundesliga_matches"]).reindex(
                columns=history_columns
            )
        )
    else:
        print("No valid Bundesliga matches were collected.")


In [ ]:
# 11. Final JSON Output Cell
final_output = {
    str(team_id): {
        "team": result["team"],
        "overall_form": result["overall_form"],
        "bundesliga_form": result["bundesliga_form"],
        "overall_matches": result["overall_matches"],
        "bundesliga_matches": result["bundesliga_matches"],
    }
    for team_id, result in form_results.items()
}

print(json.dumps(final_output, ensure_ascii=False, indent=2))


In [ ]:
# 12. Close Chrome
if driver is not None:
    # Handle expected failures with a clear, actionable message.
    try:
        driver.quit()
        print("Chrome driver closed.")
    except Exception as exc:
        print(f"Chrome driver shutdown warning: {exc}")
    finally:
        driver = None
else:
    print("Chrome driver is already closed or was not initialized.")
